In [4]:
import requests
import pandas

In [2]:
from dotenv import load_dotenv
import os

# .env lives one directory up (db/.env), notebook is in db/seed/
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
HOST              = "127.0.0.1"
PORT              = "5432"

print(f"Loaded → user={POSTGRES_USER}, db={POSTGRES_DB}")


Loaded → user=admin, db=dbt_db


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, Integer, text, String
import os 

# 1. Define connection parameters
USER = POSTGRES_USER
PASSWORD = POSTGRES_PASSWORD
HOST = HOST
PORT = "5432"  # Default PostgreSQL port
DB_NAME = POSTGRES_DB



dfs={}

## GET DICTIONARIES OF DATAFRAMES FROM FOLDER ITEMS: EXLUDE MIGRATION FILE AND README.MD
def load_dfs():
    folder_path = os.getcwd()
    curr_file = os.getcwd()
    print(curr_file + '\migration.ipynb')
    NOTEBOOK_NAME = "migration.ipynb" 
    README = "README.md"

    

    for item in os.listdir(folder_path):
            if item != NOTEBOOK_NAME and item != README:
                df_name = item.removesuffix(".csv")
                dfs[df_name] = pd.read_csv(item)
                #print(dfs.keys())
                #print(item)
    return dfs


def execute_migration(table):
    # 2. Choose your driver (psycopg2 is the most common)
    # Standard synchronous connection
    url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

    # 3. Create the engine
    engine = create_engine(url)

    # 4. Test the connection
    with engine.connect() as conn:
        print("Successfully connected to PostgreSQL!")

        df_to_upload = dfs[table]

        df_to_upload.to_sql(
            name=table,          # Destination table name
            con=conn,                 # Pass your 'conn' connection object here
            schema='bronze',     # Target database schema
            if_exists="replace",      # 'fail', 'replace', or 'append'
            index=False               # Set to False to skip saving the dataframe index row
        )

        conn.commit()
        print("All tables successfully committed to the database!")



#dfs = load_dfs()
#for i in dfs.keys():
#    execute_migration(i)




In [ ]:
df = pd.read_csv('order_products__prior.csv')
print(df.info())

   order_id  product_id  add_to_cart_order  reordered
0         2       33120                  1          1
1         2       28985                  2          1
2         2        9327                  3          0
3         2       45918                  4          1
4         2       30035                  5          0


In [5]:
print(df.describe())

           order_id    product_id  add_to_cart_order     reordered
count  3.243449e+07  3.243449e+07       3.243449e+07  3.243449e+07
mean   1.710749e+06  2.557634e+04       8.351076e+00  5.896975e-01
std    9.873007e+05  1.409669e+04       7.126671e+00  4.918886e-01
min    2.000000e+00  1.000000e+00       1.000000e+00  0.000000e+00
25%    8.559430e+05  1.353000e+04       3.000000e+00  0.000000e+00
50%    1.711048e+06  2.525600e+04       6.000000e+00  1.000000e+00
75%    2.565514e+06  3.793500e+04       1.100000e+01  1.000000e+00
max    3.421083e+06  4.968800e+04       1.450000e+02  1.000000e+00


In [ ]:
url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

# 3. Create the engine
engine = create_engine(url)

# 4. Test the connection
with engine.connect() as conn:
    print("Successfully connected to PostgreSQL!")


    df.to_sql(
    name='order_products__prior',          # Destination table name
    con=conn,                 # Pass your 'conn' connection object here
    schema='bronze',     # Target database schema
    if_exists="replace",      # 'fail', 'replace', or 'append
    index=False,              # Set to False to skip saving the dataframe index row
    chunksize=500000,
    method="multi"
    )

    conn.commit()

Successfully connected to PostgreSQL!


In [12]:
import io
import math
from tqdm import tqdm
from sqlalchemy import create_engine

USER = "admin"
PASSWORD = "admin"
HOST = "127.0.0.1"
PORT = "5432"  # Default PostgreSQL port
DB_NAME = "dbt_db"

df = pd.read_csv('products.csv')
print(df.describe())

url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(url)


def copy_from_dataframe(conn, df, table, schema, chunk_size=500_000):
    raw_conn = conn.connection
    cursor = raw_conn.cursor()

    # Drop and recreate table using pandas DDL trick
    cursor.execute(f'DROP TABLE IF EXISTS {schema}."{table}"')
    raw_conn.commit()

    # Create empty table first
    df.head(0).to_sql(table, con=conn, schema=schema, index=False, if_exists='replace')
    raw_conn.commit()

    total_chunks = math.ceil(len(df) / chunk_size)

    with tqdm(total=len(df), unit="rows", desc=f"Loading {table}") as pbar:
        for i in range(total_chunks):
            chunk = df.iloc[i * chunk_size : (i + 1) * chunk_size]

            buffer = io.StringIO()
            chunk.to_csv(buffer, index=False, header=False)
            buffer.seek(0)

            cursor.copy_expert(
                f'COPY {schema}."{table}" FROM STDIN WITH (FORMAT CSV)',
                buffer
            )
            raw_conn.commit()
            pbar.update(len(chunk))

    cursor.close()
    print(f"✓ Done — {len(df):,} rows loaded into {schema}.{table}")

with engine.connect() as conn:
    copy_from_dataframe(conn, df, 'products', 'raw')

         product_id      aisle_id  department_id
count  49688.000000  49688.000000   49688.000000
mean   24844.500000     67.769582      11.728687
std    14343.834425     38.316162       5.850410
min        1.000000      1.000000       1.000000
25%    12422.750000     35.000000       7.000000
50%    24844.500000     69.000000      13.000000
75%    37266.250000    100.000000      17.000000
max    49688.000000    134.000000      21.000000


Loading products: 100%|██████████| 49688/49688 [00:00<00:00, 340226.20rows/s]

✓ Done — 49,688 rows loaded into raw.products
